# 05. Content-модели: TF-IDF и эмбеддинги

Контекст — `решения.md`: 3.2 (модели 2 и 3), 3.1 (оценка), 1.5 (книги без оценок).

- Та же оценка, что в `03_baselines`: подвыборка 20% пользователей (seed 42), валидация — 2021 год, метрики @10 из `ml/src/eval/`
  по разрезам «все / старые / новые».
- **Профиль пользователя** — среднее векторов книг, оценённых в обучении на 4–5 (если таких нет — всех оценённых).
  Рекомендации — ближайшие к профилю книги, кроме уже оценённых (`ml/src/content/recommend.py`). Параметров нет, подбора нет.
- **Векторы книг:** TF-IDF по полю `text` (слова, 200 тыс. признаков) и эмбеддинги `e5-small` — модель, выбранная для поиска
  (`04_search_models`); для сравнения — `e5-base`.
- **Из чего выбирают модели.** Popularity и ALS — только книги с оценками в обучении (других они не знают).
  Content-модели — **весь каталог** (408 тыс. книг без 0,6% книг «из одного названия», см. 04), как на сайте. Если ограничить их
  книгами подвыборки, результат завышается: туда попадают только книги, которые потом кто-то оценит (раздел 3).
- Coverage@10 — число разных рекомендованных книг, делённое на число книг обучения (83,6 тыс.), у всех моделей одинаково.

In [1]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from implicit.cpu.als import AlternatingLeastSquares
from rectools import Columns
from rectools.models import ImplicitALSWrapperModel, PopularModel

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))
from content.embeddings import catalog, load_books
from content.recommend import recommend, tfidf_vectors
from eval.data import load_ratings, subsample_users, time_split, to_dataset
from eval.metrics import K, eval_users, evaluate, paired_bootstrap, with_ci
from eval.search import informative

REPORTS = Path.cwd().parent / "reports" / "metrics"
pd.options.display.float_format = "{:.4f}".format
FRAC, SEED, EMB = 0.2, 42, "e5-small"

sub = subsample_users(load_ratings(), FRAC, seed=SEED)
train, val, test = time_split(sub)
users = eval_users(train, val)

books = load_books(["parent_asin", "author", "text", "features", "categories"])
ids_all = books["parent_asin"].to_numpy()
allowed_all = informative(books)                                       # без книг «из одного названия» (04)
train_items = train[Columns.Item].astype(str).unique()
print(f"Пользователей для проверки: {len(users):,}; книг обучения: {len(train_items):,}; "
      f"книг, из которых выбирают content-модели: {allowed_all.sum():,}")

Пользователей для проверки: 21,350; книг обучения: 83,639; книг, из которых выбирают content-модели: 405,555


## 1. Модели

In [2]:
results, per_user, recos, timing = [], {}, {}, []

def run(name, make_reco, store=True):
    t0 = time.time()
    reco = make_reco()
    seconds = time.time() - t0
    table, pu = evaluate(reco, train, val, users)
    if store:
        results.append(with_ci(table, pu).assign(модель=name))
        per_user[name] = pu
        recos[name] = reco
        timing.append({"модель": name, "обучение + рекомендации, с": round(seconds, 1)})
    print(f"{name}: {seconds:.0f} с, NDCG@10 все / старые / новые: " + " / ".join(f"{v:.4f}" for v in table["NDCG@10"]))
    return table

ds = to_dataset(train, weighted=False)

def fit_reco(model):
    model.fit(ds)
    return model.recommend(users, ds, k=K, filter_viewed=True, on_unsupported_targets="ignore")

run("Popularity", lambda: fit_reco(PopularModel(popularity="n_users")))
run("ALS", lambda: fit_reco(ImplicitALSWrapperModel(
    AlternatingLeastSquares(factors=64, iterations=15, random_state=SEED, num_threads=0))))

X_all, _ = tfidf_vectors(books["text"])
run("Content TF-IDF", lambda: recommend(X_all, ids_all, train, users, k=K, allowed=allowed_all))
run(f"Content {EMB}", lambda: recommend(np.asarray(catalog(EMB)), ids_all, train, users, k=K, allowed=allowed_all))
_ = run("Content e5-base", lambda: recommend(np.asarray(catalog("e5-base")), ids_all, train, users, k=K, allowed=allowed_all))

Popularity: 1 с, NDCG@10 все / старые / новые: 0.0024 / 0.0035 / 0.0000


ALS: 14 с, NDCG@10 все / старые / новые: 0.0042 / 0.0061 / 0.0000


Content TF-IDF: 340 с, NDCG@10 все / старые / новые: 0.0111 / 0.0064 / 0.0171


Content e5-small: 23 с, NDCG@10 все / старые / новые: 0.0085 / 0.0047 / 0.0135


Content e5-base: 43 с, NDCG@10 все / старые / новые: 0.0105 / 0.0061 / 0.0163


## 2. Результаты на валидации

In [3]:
res = pd.concat(results, ignore_index=True)
cols = ["модель", "разрез", "пользователей", "NDCG@10", "NDCG@10 95% ДИ", "Recall@10", "Precision@10", "Coverage@10", "Novelty@10"]
res = res[cols]
res.to_csv(REPORTS / "content_val.csv", index=False, encoding="utf-8")
display(res.style.format({"пользователей": "{:,}", "NDCG@10": "{:.4f}", "Recall@10": "{:.4f}", "Precision@10": "{:.4f}",
                          "Coverage@10": "{:.4f}", "Novelty@10": "{:.2f}"}, na_rep="").hide(axis="index"))
display(pd.DataFrame(timing).style.hide(axis="index"))

модель,разрез,пользователей,NDCG@10,NDCG@10 95% ДИ,Recall@10,Precision@10,Coverage@10,Novelty@10
Popularity,все,"21,350",0.0024,0.0020–0.0028,0.0051,0.0010,0.0002,6.82
Popularity,старые,"15,945",0.0035,0.0030–0.0041,0.0076,0.0013,,
Popularity,новые,"9,396",0.0000,0.0000–0.0000,0.0000,0.0000,,
ALS,все,"21,350",0.0042,0.0037–0.0047,0.0070,0.0018,0.0106,8.84
ALS,старые,"15,945",0.0061,0.0054–0.0069,0.0106,0.0024,,
ALS,новые,"9,396",0.0000,0.0000–0.0000,0.0000,0.0000,,
Content TF-IDF,все,"21,350",0.0111,0.0102–0.0120,0.0197,0.0041,0.7999,15.63
Content TF-IDF,старые,"15,945",0.0064,0.0056–0.0073,0.0118,0.0021,,
Content TF-IDF,новые,"9,396",0.0171,0.0153–0.0189,0.0318,0.0057,,
Content e5-small,все,"21,350",0.0085,0.0076–0.0093,0.0144,0.0029,0.6601,15.42


модель,"обучение + рекомендации, с"
Popularity,0.700000
ALS,14.400000
Content TF-IDF,340.200000
Content e5-small,22.700000
Content e5-base,43.400000


In [4]:
# Разница NDCG@10 — парный bootstrap по одним и тем же пользователям
emb = f"Content {EMB}"
pairs = [(emb, "ALS", "все"), (emb, "ALS", "старые"), (emb, "ALS", "новые"), ("Content TF-IDF", "ALS", "все"),
         ("Content TF-IDF", "ALS", "старые"), (emb, "Content TF-IDF", "все"), (emb, "Content TF-IDF", "новые"),
         ("Content e5-base", emb, "все")]
rows_ = []
for a, b, s in pairs:
    r = paired_bootstrap(per_user[a][s], per_user[b][s])
    rows_.append({"сравнение": f"{a} − {b}", "разрез": s, "разница NDCG@10": r["разница"],
                  "95% ДИ": f"{r['low']:+.4f} … {r['high']:+.4f}", "значимо": "да" if r["low"] > 0 or r["high"] < 0 else "нет"})
pd.DataFrame(rows_).style.format({"разница NDCG@10": "{:+.4f}"}).hide(axis="index")

сравнение,разрез,разница NDCG@10,95% ДИ,значимо
Content e5-small − ALS,все,+0.0043,+0.0032 … +0.0052,да
Content e5-small − ALS,старые,-0.0015,-0.0026 … -0.0003,да
Content e5-small − ALS,новые,+0.0135,+0.0118 … +0.0151,да
Content TF-IDF − ALS,все,+0.0069,+0.0058 … +0.0079,да
Content TF-IDF − ALS,старые,+0.0003,-0.0009 … +0.0014,нет
Content e5-small − Content TF-IDF,все,-0.0026,-0.0034 … -0.0017,да
Content e5-small − Content TF-IDF,новые,-0.0036,-0.0053 … -0.0019,да
Content e5-base − Content e5-small,все,+0.0020,+0.0013 … +0.0027,да


**Вывод.**
- **На «новых» книгах content-модели лучше нуля** — главное, что требовалось: NDCG@10 TF-IDF 0,0171, e5-small 0,0135, e5-base 0,0163
  при нуле у Popularity и ALS (у них таких книг нет). Больше половины рекомендаций content-моделей (57–63%) — книги без единой
  оценки до 2021 года.
- **По всем книгам content-модели тоже обходят ALS по умолчанию:** TF-IDF 0,0111, e5-base 0,0105, e5-small 0,0085 против 0,0042
  (все разницы значимы). Но выигрыш — за счёт «новых» книг.
- **На «старых» книгах ALS не хуже:** ALS 0,0061, TF-IDF 0,0064 (разница незначима), e5-small 0,0047 (значимо хуже ALS).
  Это и есть довод за гибрид (3.3): коллаборативная часть — для книг с историей, content — для новых.
- **TF-IDF лучше эмбеддингов** в рекомендациях (e5-small − TF-IDF: −0,0026, значимо). Причина — в разделе 4: попадания content-моделей
  почти всегда книги знакомых авторов и серий, а их точнее ловят совпадающие слова (имя героя, название серии), чем смысл.
- **e5-base лучше e5-small** на +0,0020 (значимо, +24%) — в отличие от поиска, где они наравне.
- Coverage и Novelty у content-моделей очень высокие (66–80% от числа книг обучения, Novelty 15,4): они рекомендуют много
  разных малоизвестных книг, включая книги без оценок.
- ALS — с параметрами по умолчанию; после подбора (Optuna) соотношение на «старых» книгах может измениться.

## 3. Проверка: из какого каталога выбирать

Если content-модели выбирают только из книг подвыборки, среди «новых» книг остаются лишь те, у которых потом наберётся
≥ 5 оценок, — это подсказка из будущего. Для сравнения — тот же расчёт на каталоге подвыборки.

In [5]:
items = np.asarray(sub[Columns.Item].cat.categories, dtype=str)
rows = pd.Series(np.arange(len(books)), index=ids_all).loc[items].to_numpy()
allowed = allowed_all[rows]
cmp = []
for name, make in [("Content TF-IDF", lambda: recommend(tfidf_vectors(books["text"].to_numpy()[rows])[0], items, train, users,
                                                       k=K, allowed=allowed)),
                   (emb, lambda: recommend(np.asarray(catalog(EMB))[rows], items, train, users, k=K, allowed=allowed))]:
    t = run(name + " (каталог подвыборки)", make, store=False)
    full = res[(res["модель"] == name)].set_index("разрез")["NDCG@10"]
    cmp.append(pd.DataFrame({"модель": name, "разрез": t["разрез"], "весь каталог": full.reindex(t["разрез"]).to_numpy(),
                             "каталог подвыборки": t["NDCG@10"].to_numpy()}))
print(f"Книг в каталоге подвыборки: {len(items):,}, из них без оценок в обучении: {len(items) - len(train_items):,}")
pd.concat(cmp).style.format(precision=4).hide(axis="index")

Content TF-IDF (каталог подвыборки): 92 с, NDCG@10 все / старые / новые: 0.0193 / 0.0110 / 0.0302


Content e5-small (каталог подвыборки): 3 с, NDCG@10 все / старые / новые: 0.0122 / 0.0071 / 0.0189
Книг в каталоге подвыборки: 89,288, из них без оценок в обучении: 5,649


модель,разрез,весь каталог,каталог подвыборки
Content TF-IDF,все,0.0111,0.0193
Content TF-IDF,старые,0.0064,0.0110
Content TF-IDF,новые,0.0171,0.0302
Content e5-small,все,0.0085,0.0122
Content e5-small,старые,0.0047,0.0071
Content e5-small,новые,0.0135,0.0189


**Вывод.** На каталоге подвыборки NDCG@10 content-моделей завышен в 1,4–1,7 раза (TF-IDF 0,0193 против 0,0111): из 5,6 тыс.
«новых» книг подвыборки каждую потом кто-то оценил, а в настоящем каталоге новых книг в десятки раз больше. Поэтому основной
результат — выбор из всего каталога. Для коллаборативных моделей этот вопрос не стоит: они выбирают только из книг обучения.

## 4. Что именно угадывают content-модели

In [6]:
author = dict(zip(ids_all, books["author"]))
hits = val[val["rating"] >= 4]
hits = pd.DataFrame({"u": hits[Columns.User].astype(str), "i": hits[Columns.Item].astype(str)}).drop_duplicates()
hits = hits[hits["u"].isin(users)]
seen = {(u, author.get(i)) for u, i in zip(train[Columns.User].astype(str), train[Columns.Item].astype(str)) if author.get(i)}
known = lambda df: np.array([(u, author.get(i)) in seen for u, i in zip(df["u"], df["i"])])
new = lambda df: ~df["i"].isin(set(train_items)).to_numpy()

rows_ = [{"модель": "валидация (все попадания)", "попаданий": len(hits), "знакомый автор, % попаданий": known(hits).mean()}]
for name, r in recos.items():
    r = pd.DataFrame({"u": r[Columns.User].astype(str), "i": r[Columns.Item].astype(str)})
    h = r.merge(hits, on=["u", "i"])
    rows_.append({"модель": name, "знакомый автор, % рекомендаций": known(r).mean(), "новые книги, % рекомендаций": new(r).mean(),
                  "попаданий": len(h), "знакомый автор, % попаданий": known(h).mean() if len(h) else np.nan})
pd.DataFrame(rows_).style.format({c: "{:.1%}" for c in ["знакомый автор, % рекомендаций", "новые книги, % рекомендаций",
                                                         "знакомый автор, % попаданий"]}, na_rep="").hide(axis="index")

модель,попаданий,"знакомый автор, % попаданий","знакомый автор, % рекомендаций","новые книги, % рекомендаций"
валидация (все попадания),51295,16.3%,,
Popularity,206,4.4%,0.8%,0.0%
ALS,385,36.1%,13.2%,0.0%
Content TF-IDF,868,89.5%,40.8%,62.8%
Content e5-small,611,83.8%,20.7%,56.6%
Content e5-base,778,84.7%,24.3%,58.1%


**Вывод.** 84–90% попаданий content-моделей — книги авторов, которых пользователь уже читал (продолжения серий, другие книги того же
автора); у ALS — 36%, а среди всех попаданий валидации таких 16%. Content-модели в этой постановке — в основном «ещё от этого
автора / следующая книга серии». Это полезно (на Amazon Books серии — значимая часть чтения), но для ленты важно:
- правило «не больше 2 книг одного автора» (3.3) срежет большую часть попаданий content-части — его вес и порог нужно проверять
  на валидации, а не ставить заранее;
- открытие нового (другие авторы и темы) content-модели почти не дают — это задача коллаборативной части и правила
  «1–2 книги из соседних тем».